# LeetCode 3880 — Minimum Absolute Difference Between Two Values bằng Python

Link đề: https://leetcode.com/problems/minimum-absolute-difference-between-two-values/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Binary Search, One Pass

Skeleton LeetCode:

```python
class Solution:
    def minAbsoluteDifference(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Biweekly Contest 179, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng `nums` chỉ gồm `0`, `1`, `2`, tìm **khoảng cách vị trí nhỏ nhất** `abs(i - j)` giữa một số `1` (ở `i`) và một số `2` (ở `j`); không có cặp nào thì trả `-1`.
2. Nhận ra đây là bài "khoảng cách gần nhất giữa hai loại phần tử", có lời giải **một lượt** bằng cách nhớ vị trí gần nhất của mỗi loại.
3. Biết dùng `abs`, `enumerate`, `bisect`, trộn hai danh sách chỉ số đã sắp xếp (hai con trỏ).
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: chỉ xét `1` đứng trước `2` (bỏ chiều ngược lại), xét cặp cùng loại, chỉ xét hai phần tử kề nhau trong mảng gốc, quên `-1`, khởi tạo sai giá trị ban đầu.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` mà mỗi phần tử chỉ là `0`, `1` hoặc `2`. Một cặp chỉ số `(i, j)` là **hợp lệ** nếu `nums[i] == 1` và `nums[j] == 2`. Trả về **giá trị nhỏ nhất** của `abs(i - j)` trên mọi cặp hợp lệ, hoặc `-1` nếu không có cặp hợp lệ nào. (`i` có thể nhỏ hơn hoặc lớn hơn `j`.)

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,0,0,2,0,1]` | `2` | Các cặp: `(0,3)` cho 3 và `(5,3)` cho 2; nhỏ nhất là 2 |
| 2 | `[1,0,1,0]` | `-1` | Mảng không có số `2` nên không có cặp hợp lệ |

Constraints: `1 <= nums.length <= 100`; `0 <= nums[i] <= 2`.

In [1]:
for nums in ([1, 0, 0, 2, 0, 1], [1, 0, 1, 0]):
    ones = [i for i, x in enumerate(nums) if x == 1]
    twos = [i for i, x in enumerate(nums) if x == 2]
    print(nums, "vị trí 1:", ones, "vị trí 2:", twos)

[1, 0, 0, 2, 0, 1] vị trí 1: [0, 5] vị trí 2: [3]
[1, 0, 1, 0] vị trí 1: [0, 2] vị trí 2: []


## 3. Kiến thức nền

### 3.1. Giá trị tuyệt đối `abs`

`abs(a - b)` là khoảng cách giữa hai số, không phụ thuộc thứ tự: `abs(3 - 5) == abs(5 - 3) == 2`.

In [2]:
print(abs(3 - 5), abs(5 - 3), abs(0))

2 2 0


### 3.2. `enumerate` và gom danh sách vị trí

`enumerate(nums)` cho `(chỉ_số, giá_trị)`. List comprehension `[i for i, x in enumerate(nums) if x == 1]` gom mọi vị trí của số `1` theo thứ tự **tăng dần**.

In [3]:
nums = [1, 0, 0, 2, 0, 1]
print([i for i, x in enumerate(nums) if x == 1], [i for i, x in enumerate(nums) if x == 2])

[0, 5] [3]


### 3.3. `float('inf')` và `min`

`float('inf')` lớn hơn mọi số, thích hợp làm giá trị khởi tạo khi tìm min. Sau khi duyệt, nếu giá trị vẫn là `inf` nghĩa là chưa tìm thấy cặp nào -> trả `-1`.

In [4]:
best = float('inf')
for d in [3, 2, 5]:
    best = min(best, d)
print(best, float('inf') == float('inf'), min(float('inf'), 7))

2 True 7


### 3.4. `bisect` — tìm vị trí chèn trong danh sách đã sắp xếp

`bisect_left(a, x)` trả về chỉ số `k` sao cho chèn `x` vào `a[k]` vẫn giữ thứ tự. Các phần tử gần `x` nhất là `a[k - 1]` (nếu `k > 0`) và `a[k]` (nếu `k < len(a)`). Đây là cách tìm "hàng xóm gần nhất" trong `O(log n)`.

In [5]:
from bisect import bisect_left
a = [3, 7, 12]
for x in [0, 5, 7, 20]:
    k = bisect_left(a, x)
    nb = [a[t] for t in (k - 1, k) if 0 <= t < len(a)]
    print(f"x={x:2d} k={k} hàng xóm gần: {nb}")

x= 0 k=0 hàng xóm gần: [3]
x= 5 k=1 hàng xóm gần: [3, 7]
x= 7 k=1 hàng xóm gần: [3, 7]
x=20 k=3 hàng xóm gần: [12]


### 3.5. Hai con trỏ trên hai danh sách đã sắp xếp

Với hai danh sách tăng dần `A` và `B`, ta dùng `p` cho `A`, `q` cho `B`: tính `abs(A[p] - B[q])`, rồi **tăng con trỏ trỏ vào giá trị nhỏ hơn** (vì tăng con trỏ trỏ vào giá trị lớn hơn chỉ làm khoảng cách xa hơn). Cách này duyệt mỗi danh sách đúng một lần.

In [6]:
A, B = [0, 5], [3]
p = q = 0
best = float('inf')
while p < len(A) and q < len(B):
    best = min(best, abs(A[p] - B[q]))
    if A[p] < B[q]:
        p += 1
    else:
        q += 1
print(best)

2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ các số `1` và `2` quan trọng; số `0` chỉ là "khoảng đệm". Hãy viết ra vị trí của mọi số `1` (danh sách `A`) và mọi số `2` (danh sách `B`). Đáp án là **khoảng cách nhỏ nhất giữa một phần tử của `A` và một phần tử của `B`**.

Tính tay `[1,0,0,2,0,1]`: `A = [0, 5]`, `B = [3]`. Khoảng cách: `|0 - 3| = 3`, `|5 - 3| = 2`. Nhỏ nhất là `2`.

Quan sát hình học: trên một trục số, nếu ta đánh dấu `1` (A) và `2` (B), cặp gần nhất phải là **hai điểm khác loại đứng cạnh nhau** (không có điểm nào khác `1`/`2` nằm giữa chúng làm "ngắt"). Nếu giữa một `1` ở `i` và một `2` ở `j` còn có một số `1` hoặc `2` khác, thì ta có thể thay bằng cặp gần hơn. Do đó chỉ cần xét các cặp **khác loại liền kề** trong dãy chỉ chứa `1`/`2`.

| vị trí | 0 | 3 | 5 |
|---|---|---|---|
| giá trị | 1 | 2 | 1 |
| cặp liền kề | (0,3) khác loại: 3 | (3,5) khác loại: 2 | |

In [7]:
nums = [1, 0, 0, 2, 0, 1]
events = [(i, x) for i, x in enumerate(nums) if x in (1, 2)]
print(events)
print([abs(events[k][0] - events[k + 1][0]) for k in range(len(events) - 1) if events[k][1] != events[k + 1][1]])

[(0, 1), (3, 2), (5, 1)]
[3, 2]


## 5. Cách 1 — Brute Force / baseline

Hai vòng lặp lồng nhau: với mọi `i`, `j`, nếu `nums[i] == 1` và `nums[j] == 2` thì cập nhật `best = min(best, abs(i - j))`. Cuối cùng trả `-1` nếu `best` vẫn là `inf`.

Nút thắt: `O(n²)` thời gian, trong khi chỉ cần quan tâm tới vị trí gần nhất của mỗi loại.

In [8]:
class SolutionBrute:
    def minAbsoluteDifference(self, nums):
        n = len(nums)
        best = float('inf')
        for i in range(n):
            if nums[i] != 1:
                continue
            for j in range(n):
                if nums[j] == 2:
                    best = min(best, abs(i - j))
        return -1 if best == float('inf') else best

print(SolutionBrute().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), SolutionBrute().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** duyệt **một lượt** từ trái sang phải, nhớ `last1` = vị trí gần nhất đã gặp của số `1` và `last2` = vị trí gần nhất đã gặp của số `2` (ban đầu `None`).

- Gặp `1` ở `i`: nếu đã từng gặp `2` thì ứng viên `i - last2`; sau đó cập nhật `last1 = i`.
- Gặp `2` ở `i`: nếu đã từng gặp `1` thì ứng viên `i - last1`; sau đó cập nhật `last2 = i`.

**Invariant:** sau khi xử lý các vị trí `0..i`, `best` là khoảng cách nhỏ nhất giữa mọi cặp (một `1`, một `2`) **đều nằm trong `0..i`**. Mỗi cặp `(1 ở p, 2 ở q)` với `p < q` được xét đúng lúc ta đến `q` (lúc đó `last1 >= p` nên khoảng cách `q - last1 <= q - p`), và cặp với `q < p` được xét lúc đến `p`. Chỉ cần vị trí **gần nhất** vì vị trí xa hơn cho khoảng cách lớn hơn.

Các cách khác: gom hai danh sách vị trí rồi hai con trỏ / `bisect` / xét các cặp khác loại liền kề.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt với `last1`, `last2`

Chính là thuật toán ở mục 6.

In [9]:
class SolutionOnePass:
    def minAbsoluteDifference(self, nums):
        last1 = last2 = None
        best = float('inf')
        for i, x in enumerate(nums):
            if x == 1:
                if last2 is not None:
                    best = min(best, i - last2)
                last1 = i
            elif x == 2:
                if last1 is not None:
                    best = min(best, i - last1)
                last2 = i
        return -1 if best == float('inf') else best

print(SolutionOnePass().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), SolutionOnePass().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Gom vị trí rồi hai con trỏ

`A` = vị trí các số `1`, `B` = vị trí các số `2` (đã tăng dần). Hai con trỏ như ở mục 3.5.

In [10]:
class SolutionTwoPointers:
    def minAbsoluteDifference(self, nums):
        A = [i for i, x in enumerate(nums) if x == 1]
        B = [i for i, x in enumerate(nums) if x == 2]
        if not A or not B:
            return -1
        p = q = 0
        best = float('inf')
        while p < len(A) and q < len(B):
            best = min(best, abs(A[p] - B[q]))
            if A[p] < B[q]:
                p += 1
            else:
                q += 1
        return best

print(SolutionTwoPointers().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), SolutionTwoPointers().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho hai danh sách vị trí.

### Cách 4 — `bisect`: tìm hàng xóm gần nhất của mỗi số `1` trong danh sách vị trí số `2`

Với mỗi vị trí `a` của số `1`, tìm trong `B` (đã tăng dần) phần tử gần `a` nhất bằng `bisect_left`.

In [11]:
from bisect import bisect_left

class SolutionBisect:
    def minAbsoluteDifference(self, nums):
        A = [i for i, x in enumerate(nums) if x == 1]
        B = [i for i, x in enumerate(nums) if x == 2]
        if not A or not B:
            return -1
        best = float('inf')
        for a in A:
            k = bisect_left(B, a)
            for t in (k - 1, k):
                if 0 <= t < len(B):
                    best = min(best, abs(a - B[t]))
        return best

print(SolutionBisect().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), SolutionBisect().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


**Độ phức tạp:** Time `O(n log n)` ở xấu nhất (`bisect` `O(log n)` cho mỗi số `1`), Space `O(n)`.

### Cách 5 — Dãy chỉ chứa `1`/`2` và các cặp khác loại liền kề

Lọc `events = [(i, x)]` chỉ giữ `x` là `1` hoặc `2`; đáp án là `min` của `i2 - i1` trên các cặp **liền kề khác loại** (`x1 != x2`). Chính là nhận xét hình học ở mục 4.

In [12]:
class SolutionAdjacent:
    def minAbsoluteDifference(self, nums):
        events = [(i, x) for i, x in enumerate(nums) if x in (1, 2)]
        gaps = [events[k + 1][0] - events[k][0]
                for k in range(len(events) - 1)
                if events[k][1] != events[k + 1][1]]
        return min(gaps) if gaps else -1

print(SolutionAdjacent().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), SolutionAdjacent().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 trên `nums = [1, 0, 0, 2, 0, 1]`:

| i | nums[i] | last1 | last2 | ứng viên | best |
|---|---|---|---|---|---|
| 0 | 1 | None -> 0 | None | chưa có `2` | inf |
| 1 | 0 | 0 | None | | inf |
| 2 | 0 | 0 | None | | inf |
| 3 | 2 | 0 | None -> 3 | `3 - 0 = 3` | 3 |
| 4 | 0 | 0 | 3 | | 3 |
| 5 | 1 | 0 -> 5 | 3 | `5 - 3 = 2` | 2 |

In [13]:
def trace_one_pass(nums):
    last1 = last2 = None
    best = float('inf')
    for i, x in enumerate(nums):
        note = ""
        if x == 1:
            if last2 is not None:
                best = min(best, i - last2)
                note = f"cặp với 2 ở {last2}: {i - last2}"
            last1 = i
        elif x == 2:
            if last1 is not None:
                best = min(best, i - last1)
                note = f"cặp với 1 ở {last1}: {i - last1}"
            last2 = i
        print(f"i={i} x={x} last1={last1} last2={last2} best={best} {note}")
    return -1 if best == float('inf') else best

assert trace_one_pass([1, 0, 0, 2, 0, 1]) == 2
assert trace_one_pass([1, 0, 1, 0]) == -1
assert trace_one_pass([2, 1]) == 1

i=0 x=1 last1=0 last2=None best=inf 
i=1 x=0 last1=0 last2=None best=inf 
i=2 x=0 last1=0 last2=None best=inf 
i=3 x=2 last1=0 last2=3 best=3 cặp với 1 ở 0: 3
i=4 x=0 last1=0 last2=3 best=3 
i=5 x=1 last1=5 last2=3 best=2 cặp với 2 ở 3: 2
i=0 x=1 last1=0 last2=None best=inf 
i=1 x=0 last1=0 last2=None best=inf 
i=2 x=1 last1=2 last2=None best=inf 
i=3 x=0 last1=2 last2=None best=inf 
i=0 x=2 last1=None last2=0 best=inf 
i=1 x=1 last1=1 last2=0 best=1 cặp với 2 ở 0: 1


In [14]:
def trace_two_pointers(A, B):
    p = q = 0
    best = float('inf')
    while p < len(A) and q < len(B):
        d = abs(A[p] - B[q])
        best = min(best, d)
        print(f"p={p} q={q}: |{A[p]} - {B[q]}| = {d}, best={best}")
        if A[p] < B[q]:
            p += 1
        else:
            q += 1
    return best

assert trace_two_pointers([0, 5], [3]) == 2

p=0 q=0: |0 - 3| = 3, best=3
p=1 q=0: |5 - 3| = 2, best=2


## 9. Code nộp LeetCode

Dùng Cách 2 (một lượt, `O(n)` thời gian, `O(1)` bộ nhớ).

In [15]:
from typing import List, Optional


class Solution:
    def minAbsoluteDifference(self, nums: List[int]) -> int:
        last1 = last2 = -1                      # -1: chưa gặp
        best = float('inf')
        for i, x in enumerate(nums):
            if x == 1:
                if last2 != -1:
                    best = min(best, i - last2)
                last1 = i
            elif x == 2:
                if last1 != -1:
                    best = min(best, i - last1)
                last2 = i
        return -1 if best == float('inf') else best


print(Solution().minAbsoluteDifference([1, 0, 0, 2, 0, 1]), Solution().minAbsoluteDifference([1, 0, 1, 0]))

2 -1


## 10. Vì sao thuật toán đúng

1. **Mỗi cặp hợp lệ được xét ở thời điểm phần tử sau của cặp.** Với cặp `(p, q)` thỏa `nums[p] = 1`, `nums[q] = 2`: nếu `p < q`, tại `i = q` ta có `last1 >= p` (vị trí `1` gần nhất tính tới `q`), nên khoảng cách `q - last1 <= q - p`. Nếu `q < p`, tại `i = p` ta có `last2 >= q` và `p - last2 <= p - q`.
2. **Không bỏ sót cặp tối ưu.** Cặp tối ưu `(p*, q*)`: ở thời điểm phần tử sau, `last` tương ứng phải đúng bằng vị trí của phần tử trước (nếu không, đã có cặp gần hơn, trái với tính tối ưu). Do đó giá trị tối ưu được `best` ghi nhận.
3. **Không đếm cặp sai.** Mỗi ứng viên `i - last*` luôn ghép một `1` với một `2` thật sự (và dương vì `last* < i`), nên `best` không bao giờ nhỏ hơn đáp án thật.
4. **Không có cặp.** Nếu thiếu hẳn `1` hoặc `2`, không có ứng viên nào được xét, `best` còn `inf` -> trả `-1`.
5. Cách 5 đúng nhờ nhận xét: cặp khác loại gần nhất phải liền kề trong dãy `1/2`, vì chèn thêm phần tử `1`/`2` ở giữa luôn tạo cặp khác loại gần hơn.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- Cách 1: `O(n²)` thời gian, `O(1)` bộ nhớ.
- Cách 2: `O(n)` thời gian, `O(1)` bộ nhớ (tối ưu).
- Cách 3, 5: `O(n)` thời gian, `O(n)` bộ nhớ cho danh sách vị trí.
- Cách 4: `O(n log n)` thời gian, `O(n)` bộ nhớ.

Với `n <= 100` mọi cách đều rất nhanh. Best/worst khác nhau không đáng kể (đều duyệt hết mảng), trừ cách 1 luôn `O(n²)` nếu không có `break`.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (không có `1`, không có `2`, kề nhau, `2` đứng trước `1`) và 400 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionOnePass(), SolutionTwoPointers(), SolutionBisect(), SolutionAdjacent(), Solution()]

fixed = [
    ([1, 0, 0, 2, 0, 1], 2),
    ([1, 0, 1, 0], -1),
    ([1, 2], 1),
    ([2, 1], 1),                    # 2 đứng trước 1 vẫn hợp lệ
    ([2, 0, 0, 1], 3),
    ([0], -1),
    ([2], -1),
    ([1], -1),
    ([1, 1, 1, 2], 1),
    ([2, 2, 0, 0, 1, 2], 1),
    ([0, 0, 0], -1),
    ([1, 0, 2, 0, 1, 0, 0, 0, 2], 2),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.minAbsoluteDifference(list(nums)) == expected, (type(s).__name__, nums)

random.seed(38800)
for _ in range(400):
    n = random.randint(1, 100)
    weights = random.choice([(1, 1, 1), (8, 1, 1), (1, 8, 1), (1, 1, 8), (5, 1, 0), (5, 0, 1)])
    nums = random.choices([0, 1, 2], weights=weights, k=n)
    outs = [s.minAbsoluteDifference(list(nums)) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, outs)
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Lật ngược mảng không đổi đáp án. (2) Đáp án luôn `>= 1` hoặc `-1` (hai vị trí khác nhau). (3) Hoán đổi vai trò `1 <-> 2` không đổi đáp án vì khoảng cách đối xứng.

In [17]:
random.seed(3880)
for _ in range(300):
    nums = [random.randint(0, 2) for _ in range(random.randint(1, 100))]
    ans = Solution().minAbsoluteDifference(nums)
    assert ans == -1 or ans >= 1
    assert Solution().minAbsoluteDifference(nums[::-1]) == ans
    swapped = [{1: 2, 2: 1}.get(x, x) for x in nums]
    assert Solution().minAbsoluteDifference(swapped) == ans
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ xét `1` đứng **trước** `2`

Đề nói `abs(i - j)` nên `2` đứng trước `1` cũng hợp lệ. Nếu chỉ cập nhật khi gặp `2` sau `1`, ta bỏ sót.

In [18]:
def wrong_one_before_two(nums):
    last1 = None
    best = float('inf')
    for i, x in enumerate(nums):
        if x == 1:
            last1 = i
        elif x == 2 and last1 is not None:                      # SAI: thiếu chiều 2 trước 1
            best = min(best, i - last1)
    return -1 if best == float('inf') else best

print(wrong_one_before_two([2, 1]), "!=", Solution().minAbsoluteDifference([2, 1]))
assert wrong_one_before_two([2, 1]) == -1 and Solution().minAbsoluteDifference([2, 1]) == 1
assert wrong_one_before_two([1, 0, 0, 2, 0, 1]) == 3 and Solution().minAbsoluteDifference([1, 0, 0, 2, 0, 1]) == 2

-1 != 1


### Lỗi 2: tính cả cặp cùng loại (`1` với `1`)

Nếu so sánh mọi cặp phần tử khác `0`, ta lấy cả `(1, 1)` hoặc `(2, 2)` làm hợp lệ.

In [19]:
def wrong_same_type(nums):
    idx = [i for i, x in enumerate(nums) if x != 0]
    best = min((b - a for a, b in zip(idx, idx[1:])), default=float('inf'))   # SAI: mọi cặp liền kề khác 0
    return -1 if best == float('inf') else best

print(wrong_same_type([1, 1]), "!=", Solution().minAbsoluteDifference([1, 1]))
assert wrong_same_type([1, 1]) == 1 and Solution().minAbsoluteDifference([1, 1]) == -1
assert wrong_same_type([1, 1, 0, 2]) == 1 and Solution().minAbsoluteDifference([1, 1, 0, 2]) == 2

1 != -1


### Lỗi 3: chỉ xét hai phần tử **kề nhau trong mảng gốc**

Cặp tốt nhất có thể cách nhau qua một số số `0`. Chỉ so `nums[i]` và `nums[i + 1]` bỏ sót.

In [20]:
def wrong_adjacent_original(nums):
    best = float('inf')
    for i in range(len(nums) - 1):
        if {nums[i], nums[i + 1]} == {1, 2}:                     # SAI: chỉ hai ô kề nhau
            best = 1
    return -1 if best == float('inf') else best

print(wrong_adjacent_original([1, 0, 0, 2]), "!=", Solution().minAbsoluteDifference([1, 0, 0, 2]))
assert wrong_adjacent_original([1, 0, 0, 2]) == -1 and Solution().minAbsoluteDifference([1, 0, 0, 2]) == 3

-1 != 3


### Lỗi 4: khởi tạo `best = 0` hoặc quên trả `-1`

Nếu `best` bắt đầu bằng `0` thì `min` luôn là `0`; nếu quên so với `inf`, ta trả `inf` (hoặc số lớn) cho trường hợp không có cặp.

In [21]:
def wrong_init_zero(nums):
    best = 0                                                      # SAI: nên là inf
    last1 = last2 = None
    for i, x in enumerate(nums):
        if x == 1:
            if last2 is not None:
                best = min(best, i - last2)
            last1 = i
        elif x == 2:
            if last1 is not None:
                best = min(best, i - last1)
            last2 = i
    return best

def wrong_no_minus_one(nums):
    r = Solution().minAbsoluteDifference(nums)
    return r if r != -1 else float('inf')                         # SAI: không trả -1

print(wrong_init_zero([1, 0, 0, 2]), wrong_no_minus_one([1, 0, 1, 0]))
assert wrong_init_zero([1, 0, 0, 2]) == 0 and Solution().minAbsoluteDifference([1, 0, 0, 2]) == 3
assert wrong_no_minus_one([1, 0, 1, 0]) == float('inf') and Solution().minAbsoluteDifference([1, 0, 1, 0]) == -1

0 inf


### Lỗi 5: dùng `last = 0` làm giá trị "chưa gặp"

Vị trí `0` là chỉ số hợp lệ. Nếu khởi tạo `last1 = last2 = 0` và kiểm tra `if last2:` (hoặc dùng `0` như "chưa gặp") thì phần tử ở chỉ số `0` bị hiểu sai. Phải dùng `None` hoặc `-1`.

In [22]:
def wrong_last_zero(nums):
    last1 = last2 = 0                                             # SAI: 0 là một chỉ số thật
    best = float('inf')
    for i, x in enumerate(nums):
        if x == 1:
            best = min(best, i - last2)
            last1 = i
        elif x == 2:
            best = min(best, i - last1)
            last2 = i
    return -1 if best == float('inf') else best

print(wrong_last_zero([0, 0, 2]), "!=", Solution().minAbsoluteDifference([0, 0, 2]))
assert wrong_last_zero([0, 0, 2]) == 2 and Solution().minAbsoluteDifference([0, 0, 2]) == -1

2 != -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khoảng cách nhỏ nhất giữa hai loại phần tử" -> duyệt một lượt, nhớ **vị trí gần nhất** của mỗi loại (`last1`, `last2`).
- Hai danh sách vị trí đã sắp xếp -> hai con trỏ hoặc `bisect` (mẫu "gần nhất").
- Khoảng cách `abs(i - j)` với `i` có thể lớn hơn hoặc nhỏ hơn `j`: nhớ xét **cả hai chiều**.
- Dùng `None`/`-1`/`inf` làm giá trị "chưa có"; tránh dùng `0` vì `0` là chỉ số hợp lệ.
- Test: thiếu một loại, hai loại đứng sát nhau, `2` trước `1`, nhiều số cùng loại ở giữa.

## 15. Biến thể của bài

1. **Shortest Distance to a Character** (821): khoảng cách tới ký tự `c` gần nhất.
2. **Minimum Distance Between Three Equal Elements**-style / **Minimum Absolute Difference** (1200): cặp gần nhất sau khi sắp xếp.
3. **Closest Pair of Points**-style: bản hình học khó hơn.
4. **Shortest Word Distance** (243, Premium): khoảng cách nhỏ nhất giữa hai từ trong mảng.
5. **Minimum Distance Between Two Numbers**-type: bản có giá trị lớn dùng `dict` vị trí.
6. Biến thể: yêu cầu **bắt buộc** `1` đứng trước `2` (chỉ một chiều), hoặc trả về **cặp chỉ số** đạt min.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | mọi cặp `(i, j)` | `O(n²)` | `O(1)` | Oracle |
| 2. Một lượt | `last1`, `last2` | `O(n)` | `O(1)` | Nộp bài |
| 3. Hai con trỏ | hai danh sách vị trí | `O(n)` | `O(n)` | Khi đã có danh sách vị trí |
| 4. `bisect` | hàng xóm gần nhất | `O(n log n)` | `O(n)` | Khi truy vấn nhiều lần |
| 5. Cặp liền kề | lọc `1/2`, xét cặp khác loại | `O(n)` | `O(n)` | Dễ hình dung nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minAbsoluteDifference([2, 0, 1, 2])` và `Solution().minAbsoluteDifference([0, 0, 2, 2])` ra gì?

**Bài 2 (tính tay):** với `nums = [1, 0, 0, 0, 2, 0, 1, 0, 0, 2]`, liệt kê vị trí `1`, vị trí `2` và khoảng cách nhỏ nhất.

**Bài 3 (điền TODO):** hoàn thành `nearest_distance(nums, a, b)` tổng quát: khoảng cách nhỏ nhất giữa một phần tử bằng `a` và một phần tử bằng `b` (`a != b`), hoặc `-1`.

**Bài 4 (tự viết lại):** viết `min_difference_one_before_two(nums)` chỉ tính cặp `i < j` với `nums[i] == 1` và `nums[j] == 2`.

**Bài 5 (mini challenge):** viết `closest_pair_indices(nums)` trả về cặp `(i, j)` (một `1` ở `i`, một `2` ở `j`) đạt khoảng cách nhỏ nhất, hoặc `None`; nếu hòa trả cặp có `min(i, j)` nhỏ nhất.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minAbsoluteDifference([2, 0, 1, 2]), Solution().minAbsoluteDifference([0, 0, 2, 2]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def nearest_distance_todo(nums, a, b):
    best = float('inf')
    # TODO: duyệt một lượt, nhớ vị trí gần nhất của a và của b
    return -1

1 -1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[2,0,1,2]` có `1` ở chỉ số 2, hai số `2` ở chỉ số 0 và 3; `[0,0,2,2]` không có `1`. Bài 2: vị trí `1` là 0 và 6; vị trí `2` là 4 và 9.

**Mức 2 (vừa):** Bài 3: thay `1` và `2` bằng `a` và `b`. Bài 4: chỉ cập nhật khi gặp `2` và đã có `last1`. Bài 5: một lượt, mỗi lần cập nhật `best` lưu thêm cặp.

**Mức 3 (gần đáp án):** Bài 1: `1` và `-1`. Bài 2: khoảng cách nhỏ nhất `2` (cặp `(6, 4)`). Bài 4: `min_difference_one_before_two([2, 1]) = -1`. Bài 5: với `[1, 0, 0, 2, 0, 1]` cặp `(5, 3)`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().minAbsoluteDifference([2, 0, 1, 2]) == 1 and Solution().minAbsoluteDifference([0, 0, 2, 2]) == -1
assert Solution().minAbsoluteDifference([1, 0, 0, 0, 2, 0, 1, 0, 0, 2]) == 2

# Bài 3
def nearest_distance(nums, a, b):
    la = lb = None
    best = float('inf')
    for i, x in enumerate(nums):
        if x == a:
            if lb is not None:
                best = min(best, i - lb)
            la = i
        elif x == b:
            if la is not None:
                best = min(best, i - la)
            lb = i
    return -1 if best == float('inf') else best
assert nearest_distance([1, 0, 0, 2, 0, 1], 1, 2) == 2
assert nearest_distance([5, 3, 9, 5], 3, 5) == 1 and nearest_distance([3, 3], 3, 5) == -1

# Bài 4
def min_difference_one_before_two(nums):
    last1 = None
    best = float('inf')
    for i, x in enumerate(nums):
        if x == 1:
            last1 = i
        elif x == 2 and last1 is not None:
            best = min(best, i - last1)
    return -1 if best == float('inf') else best
assert min_difference_one_before_two([2, 1]) == -1 and min_difference_one_before_two([1, 0, 0, 2, 0, 1]) == 3
assert min_difference_one_before_two([1, 1, 2]) == 1

# Bài 5
def closest_pair_indices(nums):
    best, pair = float('inf'), None
    last1 = last2 = None
    for i, x in enumerate(nums):
        if x == 1:
            if last2 is not None and i - last2 < best:
                best, pair = i - last2, (i, last2)
            last1 = i
        elif x == 2:
            if last1 is not None and i - last1 < best:
                best, pair = i - last1, (last1, i)
            last2 = i
    return pair
assert closest_pair_indices([1, 0, 0, 2, 0, 1]) == (5, 3)
assert closest_pair_indices([1, 0, 1, 0]) is None and closest_pair_indices([1, 2]) == (0, 1)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "khoảng cách nhỏ nhất giữa hai loại phần tử":**

```python
last1 = last2 = -1
best = float('inf')
for i, x in enumerate(nums):
    if x == 1:
        if last2 != -1: best = min(best, i - last2)
        last1 = i
    elif x == 2:
        if last1 != -1: best = min(best, i - last1)
        last2 = i
return -1 if best == float('inf') else best
```

**Checklist trước khi submit:**

- [ ] Xét **cả hai chiều** (`1` trước `2` và `2` trước `1`)?
- [ ] Chỉ ghép `1` với `2` (không ghép cùng loại, không đụng `0`)?
- [ ] Giá trị "chưa gặp" là `-1`/`None` (không phải `0`) và `best` khởi tạo `inf`?
- [ ] Trả `-1` khi không có cặp hợp lệ?
- [ ] Đã thử `[1,0,0,2,0,1]`, `[1,0,1,0]`, `[2,1]`, `[0]`?
- [ ] Chữ ký đúng `minAbsoluteDifference(self, nums: List[int]) -> int`?